In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import numpy as np
import os

# 1. Load dữ liệu
app_df = pd.read_csv("../data/processed/application_clean.csv")
bureau_df = pd.read_csv("../data/processed/bureau_summary.csv")
prev_df = pd.read_csv("../data/processed/previous_application_summary.csv")

df = app_df.merge(bureau_df, on="sk_id_curr", how="left") \
           .merge(prev_df, on="sk_id_curr", how="left")

df["external_score_band"] = pd.cut(
    df["ext_source_2"], bins=[-1, 0.25, 0.50, 0.75, 1.0],
    labels=["Very Low", "Low", "Medium", "High"]
)

def get_overdue_band(row):
    if pd.isna(row["previous_credit_count"]): return "0. Thin File"
    if row["overdue_credit_count"] == 0: return "1. No Overdue"
    if row["overdue_credit_count"] <= 2: return "2. 1-2 Overdue"
    return "3. 3+ Overdue"

df["overdue_history_band"] = df.apply(get_overdue_band, axis=1)

# Cấu hình màu sắc & style
sns.set_theme(style="white")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

COLOR_BG = "#F4F7F6"
COLOR_CARD = "#FFFFFF"
COLOR_TEXT_MAIN = "#2C3E50"
COLOR_TEXT_SUB = "#7F8C8D"
COLOR_RED = "#E74C3C"
COLOR_BLUE = "#3498DB"
COLOR_ORANGE = "#F39C12"

if not os.path.exists("../dashboard"):
    os.makedirs("../dashboard")

def draw_kpi_card(ax, x, y, width, height, title, value, is_risk=False):
    shadow = patches.FancyBboxPatch((x+0.02, y-0.02), width, height, 
                                     boxstyle="round,pad=0.05,rounding_size=0.1", 
                                     ec="none", fc="#E0E0E0", alpha=0.5, zorder=1)
    card = patches.FancyBboxPatch((x, y), width, height, 
                                   boxstyle="round,pad=0.05,rounding_size=0.1", 
                                   ec="none", fc=COLOR_CARD, zorder=2)
    ax.add_patch(shadow)
    ax.add_patch(card)
    
    ax.text(x + width/2, y + height*0.7, title, ha='center', va='center', 
            fontsize=10, color=COLOR_TEXT_SUB, fontweight='bold', zorder=3)
    
    text_color = COLOR_RED if is_risk else COLOR_TEXT_MAIN
    ax.text(x + width/2, y + height*0.3, value, ha='center', va='center', 
            fontsize=24, color=text_color, fontweight='bold', zorder=3)

# ==========================================
# PAGE 1: PORTFOLIO OVERVIEW
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor=COLOR_BG)
# SỬA LỖI 1: Đẩy tiêu đề lên cao hơn (y=1.05) và thêm khoảng trống (pad)
fig.suptitle("Page 1: Portfolio Overview", fontsize=22, fontweight='bold', y=1.05, color=COLOR_TEXT_MAIN)

ax_kpi = axes[0, 0]
ax_kpi.set_facecolor(COLOR_BG)
ax_kpi.axis('off')
ax_kpi.set_xlim(0, 2)
ax_kpi.set_ylim(0, 2)

total_apps = len(df)
total_credit = df["amt_credit"].sum() / 1e9
default_rate = df["target"].mean() * 100
defaulted_exposure = df[df["target"] == 1]["amt_credit"].sum() / 1e9

draw_kpi_card(ax_kpi, 0, 1.1, 0.9, 0.8, "TOTAL APPLICATIONS", f"{total_apps:,}")
draw_kpi_card(ax_kpi, 1.1, 1.1, 0.9, 0.8, "TOTAL CREDIT EXPOSURE", f"{total_credit:.1f} Bn")
draw_kpi_card(ax_kpi, 0, 0.1, 0.9, 0.8, "OBSERVED DEFAULT RATE", f"{default_rate:.2f}%", is_risk=True)
draw_kpi_card(ax_kpi, 1.1, 0.1, 0.9, 0.8, "DEFAULTED EXPOSURE", f"{defaulted_exposure:.1f} Bn", is_risk=True)

ax_pie = axes[0, 1]
ax_pie.set_facecolor(COLOR_BG)
contract_loss = df[df["target"] == 1].groupby("name_contract_type", observed=True)["amt_credit"].sum()

wedges, texts, autotexts = ax_pie.pie(
    contract_loss, labels=None, autopct='%1.1f%%', startangle=90, 
    colors=[COLOR_BLUE, COLOR_ORANGE], 
    wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2),
    pctdistance=0.75, textprops=dict(color="white", weight="bold", fontsize=11)
)
ax_pie.legend(wedges, contract_loss.index, title="Contract Type", loc="center left", bbox_to_anchor=(0.9, 0, 0.5, 1), frameon=False, fontsize=12)
# SỬA LỖI 1: Thêm pad=20 để tiêu đề Donut không bị đè
ax_pie.set_title("Defaulted Exposure by Contract Type", fontweight='bold', fontsize=14, color=COLOR_TEXT_MAIN, pad=20)
centre_circle = plt.Circle((0,0),0.55,fc=COLOR_BG)
ax_pie.add_artist(centre_circle)

ax_bar = axes[1, 0]
ax_bar.set_facecolor(COLOR_BG)
df_age = df[(df["age"] >= 18) & (df["age"] <= 100)].copy()
age_order = ["<25", "25-34", "35-44", "45-54", "55+"]
age_default = df_age.groupby("age_band", observed=True)["target"].mean().reindex(age_order) * 100

bars = ax_bar.bar(age_default.index, age_default.values, color=COLOR_BLUE, width=0.6, alpha=0.9, zorder=3)
max_idx = np.argmax(age_default.values)
bars[max_idx].set_color(COLOR_RED)

ax_bar.axhline(default_rate, color=COLOR_RED, linestyle='--', linewidth=2, label=f'Benchmark ({default_rate:.2f}%)', zorder=2)
# SỬA LỖI 2: Thêm pad=15 để tiêu đề không dính vào thẻ KPI phía trên
ax_bar.set_title("Default Rate by Age Band", fontweight='bold', fontsize=14, color=COLOR_TEXT_MAIN, loc='left', pad=15)
ax_bar.set_ylabel("Default Rate (%)", color=COLOR_TEXT_SUB)
ax_bar.set_xlabel("")
ax_bar.legend(frameon=False, loc='upper right')
ax_bar.set_ylim(0, max(age_default.values) * 1.2)

ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)
ax_bar.spines['left'].set_color('#DDDDDD')
ax_bar.spines['bottom'].set_color('#DDDDDD')
ax_bar.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)
ax_bar.tick_params(colors=COLOR_TEXT_SUB)

axes[1, 1].axis('off')

# SỬA LỖI: Dùng tight_layout với rect để chừa khoảng trống cho suptitle
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("../dashboard/Page1_Portfolio_Overview.png", dpi=300, bbox_inches='tight', facecolor=COLOR_BG)
plt.close()


# ==========================================
# PAGE 2: THE RISK TRAP
# ==========================================
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [1.5, 1]}, facecolor=COLOR_BG)
# SỬA LỖI 1: Đẩy tiêu đề chính lên y=1.05
fig.suptitle("Page 2: The Risk Trap - Rate vs. Concentration", fontsize=22, fontweight='bold', y=1.05, color=COLOR_TEXT_MAIN)

q_data = df.groupby("credit_income_band", observed=True).agg(
    default_rate=("target", "mean"),
    defaulted_exp=("amt_credit", lambda x: df.loc[x.index, "target"].eq(1).mul(x).sum() / 1e9)
).reset_index()
q_data["default_rate"] *= 100

ax1 = axes[0]
ax1.set_facecolor(COLOR_BG)
ax2 = ax1.twinx()

bars = ax1.bar(q_data["credit_income_band"], q_data["default_rate"], color=COLOR_ORANGE, width=0.5, alpha=0.8, zorder=2, label="Default Rate")
bars[3].set_color(COLOR_RED)
ax1.set_ylabel("Default Rate (%)", color=COLOR_ORANGE, fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=COLOR_ORANGE)

ax2.plot(q_data["credit_income_band"], q_data["defaulted_exp"], color=COLOR_BLUE, marker="o", markersize=10, linewidth=4, zorder=3, label="Defaulted Exposure")
ax2.set_ylabel("Defaulted Exposure (Bn)", color=COLOR_BLUE, fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor=COLOR_BLUE)

# SỬA LỖI 3: Đẩy số liệu lên cao hơn một chút (+0.4) để không đè lên cột
for i, val in enumerate(q_data["defaulted_exp"]):
    ax2.text(i, val + 0.4, f"{val:.1f} Bn", ha='center', va='bottom', color=COLOR_BLUE, fontweight='bold', fontsize=11)

# SỬA LỖI 2: Thêm pad=20 cho tiêu đề chart
ax1.set_title("The Q4 Trap: Lower Default Rate, Highest Financial Loss", fontweight='bold', loc='left', fontsize=14, color=COLOR_TEXT_MAIN, pad=20)
ax1.set_xlabel("")

ax1.spines['top'].set_visible(False)
ax1.spines['left'].set_color('#DDDDDD')
ax1.spines['bottom'].set_color('#DDDDDD')
ax1.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_color('#DDDDDD')

ax_table = axes[1]
ax_table.set_facecolor(COLOR_BG)
ax_table.axis('off')
table_data = df[df["credit_income_band"].isin(["Q3", "Q4 - Highest"])].groupby(
    ["credit_income_band", "external_score_band"], observed=True
).agg(
    Applications=("sk_id_curr", "count"),
    Avg_Loan_Size=("amt_credit", lambda x: f"{x.mean():,.0f}"),
    Default_Rate=("target", lambda x: f"{x.mean()*100:.2f}%"),
    Defaulted_Exposure=("amt_credit", lambda x: f"{df.loc[x.index, 'target'].eq(1).mul(x).sum()/1e9:.2f} Bn")
).reset_index()

num_cols = len(table_data.columns)
table = ax_table.table(cellText=table_data.values, colLabels=table_data.columns, 
                      cellLoc='center', loc='center', colColours=[COLOR_BLUE]*num_cols)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor('#DDDDDD')
    if row == 0:
        cell.set_text_props(color='white', fontweight='bold')
        cell.set_height(0.15)
    else:
        cell.set_facecolor(COLOR_CARD)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("../dashboard/Page2_The_Risk_Trap.png", dpi=300, bbox_inches='tight', facecolor=COLOR_BG)
plt.close()


# ==========================================
# PAGE 3: DANGER ZONES
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor=COLOR_BG)
fig.suptitle("Page 3: Danger Zones - Early Warning Signals", fontsize=22, fontweight='bold', y=1.05, color=COLOR_TEXT_MAIN)

# Chart 1: External Score
ax_score = axes[0, 0]
ax_score.set_facecolor(COLOR_BG)
score_default = df.groupby("external_score_band", observed=True)["target"].mean() * 100
colors_score = ['#FADBD8', '#F1948A', '#E74C3C', '#922B21']
ax_score.bar(score_default.index, score_default.values, color=colors_score, width=0.6, zorder=3)
ax_score.axhline(default_rate, color='black', linestyle='--', linewidth=2, label='Benchmark')
ax_score.set_title("Default Rate by External Score", fontweight='bold', fontsize=14, color=COLOR_TEXT_MAIN, loc='left', pad=15)
ax_score.set_ylabel("Default Rate (%)", color=COLOR_TEXT_SUB)
ax_score.set_xlabel("")
ax_score.legend(frameon=False)
ax_score.spines['top'].set_visible(False)
ax_score.spines['right'].set_visible(False)
ax_score.spines['left'].set_color('#DDDDDD')
ax_score.spines['bottom'].set_color('#DDDDDD')
ax_score.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)
for i, v in enumerate(score_default.values):
    ax_score.text(i, v + 0.5, f"{v:.1f}%", ha='center', fontweight='bold', color=COLOR_TEXT_MAIN)

# Chart 2: Overdue History
ax_overdue = axes[0, 1]
ax_overdue.set_facecolor(COLOR_BG)
overdue_default = df.groupby("overdue_history_band", observed=True)["target"].mean() * 100
colors_overdue = ['#FDEBD0', '#F5B041', '#E67E22', '#D35400']
ax_overdue.bar(overdue_default.index, overdue_default.values, color=colors_overdue, width=0.6, zorder=3)
ax_overdue.axhline(default_rate, color='black', linestyle='--', linewidth=2, label='Benchmark')

# SỬA LỖI CỘT 57.6%: Thêm log scale hoặc giới hạn trục Y để các cột khác dễ nhìn hơn
# Ở đây mình dùng cách giới hạn trục Y ở mức 65% và thêm text cho cột cao nhất
ax_overdue.set_ylim(0, 65) 

ax_overdue.set_title("Default Rate by Overdue History", fontweight='bold', fontsize=14, color=COLOR_TEXT_MAIN, loc='left', pad=15)
ax_overdue.set_ylabel("Default Rate (%)", color=COLOR_TEXT_SUB)
ax_overdue.set_xlabel("")
ax_overdue.legend(frameon=False, loc='upper left')
ax_overdue.spines['top'].set_visible(False)
ax_overdue.spines['right'].set_visible(False)
ax_overdue.spines['left'].set_color('#DDDDDD')
ax_overdue.spines['bottom'].set_color('#DDDDDD')
ax_overdue.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)
for i, v in enumerate(overdue_default.values):
    ax_overdue.text(i, v + 1.5, f"{v:.1f}%", ha='center', fontweight='bold', color=COLOR_TEXT_MAIN)

axes[1, 0].axis('off')
axes[1, 1].axis('off')

# Heatmap
df["refusal_band_calc"] = df["previous_refusal_rate"].apply(
    lambda x: "No History" if pd.isna(x) else ("0% Refused" if x == 0 else ("1-49% Refused" if x < 0.5 else "50%+ Refused"))
)
pivot_table = df.pivot_table(index="refusal_band_calc", columns="overdue_history_band", 
                             values="target", aggfunc="mean", observed=True) * 100

ax_heat = fig.add_axes([0.2, 0.05, 0.6, 0.35]) 
ax_heat.set_facecolor(COLOR_BG)
sns.heatmap(pivot_table, annot=True, fmt=".1f", cmap="Reds", ax=ax_heat, cbar=False, 
            linewidths=3, linecolor=COLOR_BG, annot_kws={"size": 14, "weight": "bold", "color": "black"})
ax_heat.set_title("Interaction: Prior Refusal x Overdue History (Default Rate %)", fontweight='bold', fontsize=14, color=COLOR_TEXT_MAIN, pad=15)
ax_heat.set_xlabel("")
ax_heat.set_ylabel("Refusal Band", fontsize=12, color=COLOR_TEXT_SUB)
ax_heat.tick_params(axis='both', labelsize=11, colors=COLOR_TEXT_SUB)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("../dashboard/Page3_Danger_Zones.png", dpi=300, bbox_inches='tight', facecolor=COLOR_BG)
plt.close()

print("Đã sửa lỗi layout và xuất lại 3 ảnh Dashboard!")

C:\Users\minh2\AppData\Local\Temp\ipykernel_14820\3839775712.py:275: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])


Đã sửa lỗi layout và xuất lại 3 ảnh Dashboard!
